# PyTorch Distributed

A comprehensive guide to PyTorch Distributed for AI/ML workloads.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

PyTorch Distributed provides the core primitives for **scaling PyTorch training and inference across multiple GPUs and machines**. It powers higher-level libraries like DeepSpeed, Ray Train, PyTorch Lightning, and many custom frameworks.

### What is it?

At a high level, **PyTorch Distributed** includes:

- The `torch.distributed` package for **collective communication** (all-reduce, broadcast, etc.).
- High-level wrappers such as **DistributedDataParallel (DDP)** and **FullyShardedDataParallel (FSDP)**.
- Utilities and launchers (e.g., `torchrun`) to start multi-process distributed jobs.

### Why use it?

Key benefits of using PyTorch Distributed:

- **Scale out training** across multiple GPUs/nodes to reduce time-to-train.
- **Larger effective batch sizes** and better hardware utilization.
- **Fine-grained control** over communication patterns and process layout.

### When to use it?

PyTorch Distributed is particularly useful when:

- You want a **low-level, flexible foundation** for custom distributed training logic.
- You’re integrating with an existing cluster/orchestration system (Slurm, Kubernetes, Ray, etc.).
- You’re building or extending frameworks that need direct access to PyTorch’s distributed APIs.

## Key Features

### Core Capabilities of PyTorch Distributed

| Feature | Description | Benefit |
|--------|-------------|---------|
| **Process groups & backends** | `init_process_group` with NCCL, Gloo, MPI, etc. | Flexible communication layer for different hardware and environments. |
| **DistributedDataParallel (DDP)** | Wraps a `nn.Module` and synchronizes gradients across ranks. | Standard, well-optimized data-parallel training for most models. |
| **FullyShardedDataParallel (FSDP)** | Shards parameters, gradients, and optimizer states across ranks. | Fit larger models and reduce per-GPU memory usage. |
| **Collective operations** | All-reduce, broadcast, all-gather, reduce-scatter, etc. | Foundation for custom distributed algorithms. |
| **Launch utilities (`torchrun`)** | Starts multi-process jobs with the right environment variables. | Easier, more robust job launches compared to older scripts. |
| **RPC & pipeline parallel APIs** | Higher-level abstractions for model parallelism and serving. | Build complex distributed systems and model architectures. |

## Architecture Overview

PyTorch Distributed typically uses a **multi-process** architecture:

```text
+-----------------+       +-----------------+       +-----------------+
|  Rank 0 (proc)  |  ...  |  Rank 1 (proc)  | ...  | Rank N-1 (proc) |
|  GPU 0 / host 0 |       |  GPU 1 / host 1 |       | GPU k / host m  |
+--------+--------+       +--------+--------+       +--------+--------+
         |                          |                          |
         +----------- NCCL / Gloo process group ---------------+
```

### Key components

1. **Ranks and world size**  
   - Each process has a **rank** (0..`world_size-1`).  
   - All processes together form the **world** (process group).

2. **Backend**  
   - **NCCL** for GPU training (recommended on NVIDIA GPUs).  
   - **Gloo** for CPU-only or simple setups.

3. **DDP / FSDP wrappers**  
   - Wrap your `nn.Module` to handle synchronization and sharding.  
   - Each process runs its own copy of the model on its own device.

4. **Launch and environment**  
   - Tools like `torchrun` set environment variables (`RANK`, `WORLD_SIZE`, `LOCAL_RANK`, etc.).  
   - `torch.distributed.init_process_group()` reads these to initialize communication.

## Installation

### Prerequisites

- Python 3.8+.
- A PyTorch build with distributed support (most official binaries qualify).
- For GPU training: CUDA toolkit and compatible NVIDIA drivers.

### Installation

PyTorch Distributed is part of **PyTorch itself**, so there is no separate package to install. Use the official PyTorch installation instructions for your platform and CUDA version, for example:

```bash
# Example only – check https://pytorch.org/get-started/locally/ for the right command.
pip install torch torchvision torchaudio
```

In [ ]:
# Quick install helper for notebooks (uncomment to run)
# !pip install torch torchvision torchaudio

## Basic Usage

### Quick start: DistributedDataParallel (DDP)

The most common pattern is:

1. Launch N processes (one per GPU) using `torchrun`.
2. In each process, initialize `torch.distributed` and wrap your model with `DistributedDataParallel`.
3. Use a `DistributedSampler` for your dataset so each rank sees a unique shard.

Below is a **minimal DDP training script** that you can save as `ddp_train.py` and launch with:

```bash
torchrun --standalone --nproc_per_node=2 ddp_train.py
```

In [ ]:
# Minimal DDP example (intended for a script, not executed here)

import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP


def setup():
    """Initialize distributed process group and return (rank, world_size, device, local_rank)."""
    dist.init_process_group(backend="nccl" if torch.cuda.is_available() else "gloo")

    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", 0))

    if torch.cuda.is_available():
        device = torch.device(f"cuda:{local_rank}")
        torch.cuda.set_device(device)
    else:
        device = torch.device("cpu")

    return rank, world_size, device, local_rank


def cleanup():
    dist.destroy_process_group()


def main():
    rank, world_size, device, local_rank = setup()

    # Simple model
    model = nn.Linear(10, 1).to(device)
    ddp_model = DDP(model, device_ids=[local_rank] if device.type == "cuda" else None)
    optimizer = optim.SGD(ddp_model.parameters(), lr=1e-3)

    for step in range(5):
        x = torch.randn(32, 10, device=device)
        y = torch.randn(32, 1, device=device)

        optimizer.zero_grad()
        outputs = ddp_model(x)
        loss = ((outputs - y) ** 2).mean()
        loss.backward()
        optimizer.step()

        if rank == 0:
            print(f"Step {step} | Loss: {loss.item():.4f}")

    cleanup()


if __name__ == "__main__":
    main()

## Advanced Features

### 1. FullyShardedDataParallel (FSDP)

- Shards model parameters, gradients, and optimizer states across ranks.  
- Reduces per-GPU memory usage significantly for large models.

### 2. Pipeline and tensor model parallelism

- Combine DDP/FSDP with pipeline parallelism or tensor parallel schemes.  
- Often used in large language model training frameworks.

### 3. RPC and distributed autograd

- `torch.distributed.rpc` provides remote procedure calls and distributed autograd.  
- Useful for model-parallel serving or custom distributed architectures.

In [ ]:
# Sketch: wrapping a model with FSDP (conceptual example)

from torch.distributed.fsdp import FullyShardedDataParallel as FSDP


def fsdp_example(model, device):
    """Wrap model with FSDP after distributed initialization (not executed here)."""
    model = model.to(device)
    fsdp_model = FSDP(model)
    return fsdp_model

## Use Cases

- **Data-parallel training of vision, NLP, and recommendation models** on multi-GPU servers.  
- **Multi-node training** of large models using DDP/FSDP as the core building block.  
- **Serving and inference** patterns that rely on RPC or custom collective operations.  
- As the foundation for higher-level frameworks like **DeepSpeed**, **Ray Train**, **Horovod**, and **Lightning**.

## Best Practices

1. **Validate single-GPU training first**  
   Get your model and training loop working on a single device before adding distributed complexity.

2. **Use `torchrun` instead of legacy launch scripts**  
   It is the recommended launcher and handles environment setup more robustly.

3. **Use `DistributedSampler` for datasets**  
   Ensure each rank sees a unique shard of the data to avoid duplication and skew.

4. **Log from rank 0 only (or selectively)**  
   Avoid cluttered logs from every rank; use `if rank == 0:` guards.

5. **Set seeds and control randomness**  
   For reproducibility, seed Python, NumPy, and PyTorch on each rank.

6. **Start small when scaling**  
   Increase world size gradually and monitor performance and stability.

## Common Pitfalls

1. **Mismatched world size and launch configuration**  
   - Symptom: `RuntimeError` about expected world size vs. actual.  
   - Fix: Ensure `--nproc_per_node` × number of nodes equals `WORLD_SIZE`.

2. **Backend or device mismatch**  
   - Symptom: NCCL errors or hangs.  
   - Fix: Use `backend="nccl"` only when GPUs and drivers are available; otherwise use `gloo`.

3. **Improper device placement**  
   - Symptom: Using the wrong GPU per rank or mixing CPU/GPU tensors.  
   - Fix: Map `LOCAL_RANK` to the correct CUDA device and consistently move tensors to that device.

4. **Dataset not sharded**  
   - Symptom: Each rank sees the full dataset, causing duplicated work and biased gradients.  
   - Fix: Use `DistributedSampler` (or equivalent) with appropriate `num_replicas` and `rank`.

5. **Silent hangs due to missing `dist.destroy_process_group()`**  
   - Symptom: Scripts never exit cleanly.  
   - Fix: Call `destroy_process_group()` during shutdown.

## Performance Optimization

### Tuning knobs

- **Batch size per GPU**: Increase if memory allows to reduce overhead.  
- **Number of workers (world size)**: More GPUs can help, but watch communication overhead.  
- **Gradient accumulation**: Use when you cannot fit the desired batch size per device.  
- **Mixed precision**: Use AMP/FP16/BF16 to improve throughput and reduce memory usage.

### Measuring performance

Track:

- **Samples/sec or tokens/sec** per GPU and global.  
- **Time per step** and per epoch.  
- **GPU utilization and memory usage** via `nvidia-smi` or monitoring tools.

Benchmark on realistic workloads to ensure scaling efficiency is acceptable as you add more GPUs or nodes.

In [ ]:
# Sketch: timing a few training steps (conceptual)

import time

steps = 10
start = time.time()
for _ in range(steps):
    # Assume a training step here (forward + backward + optimizer.step)
    pass
end = time.time()

print("Average seconds per step:", (end - start) / steps)

## Production Deployment

PyTorch Distributed is agnostic to how you manage processes and machines. Common options:

1. **Single-node, multi-GPU**  
   - Launch with `torchrun --standalone --nproc_per_node=N train.py`.  
   - Use for development and smaller-scale production jobs.

2. **Multi-node clusters (Slurm, Kubernetes, etc.)**  
   - Use your scheduler to start one `torchrun` per node with the right `--nnodes` and `--node_rank`.  
   - Ensure nodes can reach each other over the network on required ports.

3. **Integration with orchestration systems**  
   - Wrap training scripts into jobs for Airflow, Argo Workflows, Ray Jobs, etc.  
   - Store checkpoints in durable storage for later evaluation and serving.

## Monitoring and Observability

- **Logs per rank**: Capture logs from each process; aggregate or focus on rank 0 for high-level metrics.  
- **System metrics**: Monitor GPU utilization, memory, CPU, and network throughput.  
- **Training metrics**: Track loss, accuracy, learning rate, and throughput per step/epoch.  
- **Cluster monitoring**: Use tools like Prometheus, Grafana, or your cloud provider’s monitoring stack to track cluster health.

## Troubleshooting

### Issue 1: Processes hang on startup

- Check that all expected ranks are launched and `WORLD_SIZE` matches.  
- Verify that the backend (NCCL/Gloo) is correctly configured and that nodes can reach each other.

### Issue 2: NCCL errors or deadlocks

- Ensure drivers and CUDA versions are compatible.  
- Try setting recommended NCCL environment variables from the PyTorch docs.  
- Test a minimal DDP script before complex workloads.

### Issue 3: Uneven workload or slow ranks

- Confirm dataset is evenly sharded and that each rank has similar work.  
- Check for straggler nodes with hardware or network issues.

### Issue 4: Out-of-memory errors

- Reduce batch size per GPU.  
- Use mixed precision.  
- Consider FSDP or external tools like DeepSpeed if model size is the main constraint.

## Comparison with Alternatives

| Aspect | PyTorch Distributed | DeepSpeed | Ray Train | Horovod |
|--------|---------------------|----------|-----------|---------|
| Level | Low-level primitives & wrappers | Higher-level optimizations on top of PyTorch | Higher-level orchestration & ecosystem | Higher-level data-parallel framework |
| Memory optimizations | DDP/FSDP (sharding via FSDP) | ZeRO, offload, fused kernels | Uses underlying PyTorch / DeepSpeed | Some optimizations, but less focused on model size |
| Ecosystem | Native PyTorch | PyTorch + DeepSpeed tools | Part of Ray (Train, Tune, Serve, Data) | MPI-centric |

### When to choose PyTorch Distributed

- You need **maximum control** over how distributed training is implemented.  
- You’re building or extending frameworks that run on top of PyTorch.  
- You want a minimal dependency stack and are comfortable managing cluster details yourself.

## Resources

### Official Documentation

- PyTorch distributed overview: https://pytorch.org/docs/stable/distributed.html
- DDP tutorial: https://pytorch.org/tutorials/intermediate/ddp_tutorial.html
- FSDP docs: https://pytorch.org/docs/stable/fsdp.html

### Tutorials and Guides

- PyTorch distributed training tutorials (official docs).  
- Community blog posts and examples for DDP/FSDP on Kubernetes, Slurm, and cloud platforms.

### Community

- PyTorch GitHub: https://github.com/pytorch/pytorch  
- PyTorch Forums: https://discuss.pytorch.org  
- Stack Overflow `pytorch` and `distributed-computing` tags.

### Related Technologies

- **DeepSpeed**, **Megatron-LM** for large-scale model training.  
- **Ray Train**, **Horovod**, **Lightning** as higher-level orchestration layers on top of PyTorch Distributed.